In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import random  
import pickle
import math
import time

device = (
    'cuda' if torch.cuda.is_available()
    else 'mps' if torch.backends.mps.is_available()
    else 'cpu'
)

batch_size = 32       
block_size = 512      
max_iters = 12500     
learning_rate = 1e-3  
eval_iters = 100 

n_embd = 512          
n_head = 8            
n_layer = 6          
dropout = 0.1

print(device)  

cuda


In [2]:
chars = " \n!\"$%&'()*+,-./0123456789:;=?ABCDEFGHIJKLMNOPQRSTUVWXYZ_`abcdefghijklmnopqrstuvwxyz¡§ª«°´º»½¿ÀÁÂÃÇÈÉÊËÍÓÔÕÚÛÜàáâãäçèéêëìíîïñòóôõöùúûüœ–—―‘’“”…"
vocab_size = len(chars) 

In [3]:
string_to_int = { ch:i for i,ch in enumerate(chars) }
int_to_string = { i:ch for i,ch in enumerate(chars) }

encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ''.join([int_to_string[i] for i in l])

In [4]:
with open("data/machado_de_assis_obra_completa.txt", 'r', encoding='utf-8') as f:
    text = f.read()

print(f"Dataset size: {len(text):,} characters")

# Converte cada caractere em seu índice no vocabulário. torch.long é o tipo esperado por nn.Embedding.
data = torch.tensor(encode(text), dtype=torch.long)

n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]
print(f"Train: {len(train_data):,} tokens | Val: {len(val_data):,} tokens")

def get_batch(split):
    """Retorna um lote aleatório de entradas e dos próximos caracteres esperados.

    Cada entrada tem formato (batch_size, block_size). O alvo é a mesma janela
    deslocada um caractere à frente, para ensinar a previsão do próximo token.
    """
    # Escolhe a partição solicitada sem misturar validação e treino
    data = train_data if split == 'train' else val_data

    # Sorteia inícios que ainda permitam obter a entrada e seu alvo deslocado
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])

    # Move os tensores para o mesmo dispositivo do modelo antes da execução
    return x.to(device), y.to(device)

Dataset size: 10,977,812 characters
Train: 9,880,030 tokens | Val: 1,097,782 tokens


In [5]:
class CausalSelfAttention(nn.Module):
    """Atenção própria multi-cabeça que só permite consultar tokens anteriores"""

    def __init__(self, n_embd, n_head):
        super().__init__()
        # Cada cabeça recebe uma fração inteira da dimensão total dos embeddings.
        assert n_embd % n_head == 0
        self.n_head = n_head
        self.head_dim = n_embd // n_head

        # Uma única projeção produz Query, Key e Value concatenados: (B, T, C) -> (B, T, 3C).
        self.c_attn = nn.Linear(n_embd, 3 * n_embd, bias=False)
        # Projeta a concatenação das cabeças de volta para a dimensão do modelo.
        self.c_proj = nn.Linear(n_embd, n_embd, bias=False)
        self.dropout = nn.Dropout(dropout)

        # Máscara triangular causal registrada como buffer (acompanha o dispositivo do modelo).
        self.register_buffer('mask', torch.tril(torch.ones(block_size, block_size))
                             .view(1, 1, block_size, block_size))

    def forward(self, x):
        # B = lote, T = comprimento da sequência e C = dimensão do embedding.
        B, T, C = x.shape

        # Separa as projeções Q, K e V, cada uma com formato (B, T, C).
        qkv = self.c_attn(x)
        q, k, v = qkv.split(C, dim=2)

        # Divide C em várias cabeças: (B, T, C) -> (B, n_head, T, head_dim).
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        # is_causal impede que uma posição veja o futuro. PyTorch usa a implementação mais eficiente disponível.
        out = F.scaled_dot_product_attention(
            q, k, v, is_causal=True, dropout_p=dropout if self.training else 0.0
        )

        # Reúne as cabeças, aplica a projeção de saída e regulariza durante o treino.
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.c_proj(out)
        out = self.dropout(out)
        return out


class FeedForward(nn.Module):
    """MLP aplicado independentemente a cada posição da sequência."""

    def __init__(self, n_embd):
        super().__init__()
        # Expande a representação em 4x, aplica não linearidade GELU e volta a n_embd.
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    """Bloco Transformer com normalização antes de cada subcamada (Pre-LayerNorm)."""

    def __init__(self, n_embd, n_head):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ffwd = FeedForward(n_embd)

    def forward(self, x):
        # As conexões residuais preservam a informação de entrada e tornam o treinamento mais estável.
        x = x + self.attn(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x


class GPTLanguageModel(nn.Module):
    """Modelo autoregressivo de caracteres baseado em blocos Transformer."""

    def __init__(self, vocab_size):
        super().__init__()
        # Embeddings identificam o caractere e sua posição no contexto.
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[TransformerBlock(n_embd, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        # Converte a representação final em um logit para cada caractere possível.
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)

        # Compartilha os pesos de entrada e saída: reduz parâmetros e costuma melhorar o modelo.
        self.token_embedding_table.weight = self.lm_head.weight

        # Inicializa pesos antes de aplicar o ajuste específico das projeções residuais.
        self.apply(self._init_weights)
        for name, p in self.named_parameters():
            if name.endswith('c_proj.weight'):
                # Inicialização GPT-2: reduz a escala acumulada das conexões residuais.
                torch.nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * n_layer))

    def _init_weights(self, module):
        """Inicializa camadas lineares e embeddings com distribuição normal pequena."""
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, index, targets=None):
        # index contém tokens inteiros com formato (B, T).
        B, T = index.shape

        tok_emb = self.token_embedding_table(index)  # (B, T) -> (B, T, C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))  # (T,) -> (T, C), difundido pelo lote
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)  # (B, T, C), em que C equivale a vocab_size

        if targets is None:
            loss = None  # Na geração só são necessários os logits do próximo token.
        else:
            # A entropia cruzada espera exemplos numa dimensão; achatamos lote e tempo.
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, index, max_new_tokens, temperature=0.8, top_k=40):
        """Acrescenta tokens amostrados autoregressivamente ao contexto inicial."""
        for _ in range(max_new_tokens):
            # O modelo foi treinado para no máximo block_size posições de contexto.
            index_cond = index[:, -block_size:]
            logits, _ = self.forward(index_cond)
            # Considera apenas a última posição; temperatura regula aleatoriedade da amostragem.
            logits = logits[:, -1, :] / temperature

            # Mantém apenas os top_k logits mais prováveis para evitar escolhas muito improváveis.
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float('-inf')

            probs = F.softmax(logits, dim=-1)
            index_next = torch.multinomial(probs, num_samples=1)
            index = torch.cat((index, index_next), dim=1)
        return index

model = GPTLanguageModel(vocab_size)
m = model.to(device)
print(f"Model parameters: {sum(p.numel() for p in m.parameters()):,}")

Model parameters: 19,238,400


In [6]:
@torch.no_grad()
def estimate_loss():
    """Estima a perda média em treino e validação sem calcular gradientes."""
    out = {}
    # eval desliga dropout para tornar a avaliação determinística em relação aos pesos.
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [7]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, betas=(0.9, 0.95), weight_decay=0.1)

def get_lr(it):
    """Calcula a taxa de aprendizado com aquecimento linear e decaimento cossenoidal."""
    warmup_iters = 100
    min_lr = learning_rate / 10
    if it < warmup_iters:
        # Aumenta gradualmente a taxa para evitar atualizações instáveis no início.
        return learning_rate * (it + 1) / warmup_iters
    # Depois do aquecimento, a curva cossenoidal diminui suavemente até min_lr.
    decay_ratio = (it - warmup_iters) / (max_iters - warmup_iters)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return min_lr + coeff * (learning_rate - min_lr)

for iter in range(max_iters):
    lr = get_lr(iter)
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr

    if iter % 100 == 0:
        losses = estimate_loss()
        print(f"step {iter:4d} | train: {losses['train']:.4f} | val: {losses['val']:.4f}")

    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

localtime = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime())
with open(f'model-{localtime}.pkl', 'wb') as f:
    pickle.dump(model, f)

step    0 | train: 5.1206 | val: 5.1130
step  100 | train: 2.4627 | val: 2.5134
step  200 | train: 2.3844 | val: 2.4381
step  300 | train: 2.0942 | val: 2.1725
step  400 | train: 1.8615 | val: 1.9675
step  500 | train: 1.7342 | val: 1.8569
step  600 | train: 1.6239 | val: 1.7659
step  700 | train: 1.5489 | val: 1.7099
step  800 | train: 1.4839 | val: 1.6425
step  900 | train: 1.4497 | val: 1.6204
step 1000 | train: 1.4142 | val: 1.5846
step 1100 | train: 1.3839 | val: 1.5691
step 1200 | train: 1.3553 | val: 1.5424
step 1300 | train: 1.3365 | val: 1.5236
step 1400 | train: 1.3218 | val: 1.5134
step 1500 | train: 1.3078 | val: 1.4958
step 1600 | train: 1.2936 | val: 1.5035
step 1700 | train: 1.2799 | val: 1.4829
step 1800 | train: 1.2682 | val: 1.4691
step 1900 | train: 1.2585 | val: 1.4605
step 2000 | train: 1.2482 | val: 1.4486
step 2100 | train: 1.2376 | val: 1.4541
step 2200 | train: 1.2293 | val: 1.4369
step 2300 | train: 1.2196 | val: 1.4365
step 2400 | train: 1.2128 | val: 1.4398


In [11]:
print(f"\nFinal loss: {loss.item():.4f}")



Final loss: 0.9495


## Geração de textos

In [8]:
prompts_machado = [
    'Ao vencedor, as batatas',
    'Deus te livre, leitor, de uma ideia fixa',
    'Algum tempo hesitei se devia abrir estas memórias pelo princípio ou pelo fim',
    'Uma noite destas, vindo da cidade para o Engenho Novo',
    'Capitu era Capitu, isto é, uma criatura mui particular',
    'Olhos de ressaca',
    'Marcela amou-me durante quinze meses e onze contos de réis',
    'A vida é uma ópera, e uma grande ópera',
    'Não tive filhos. Não transmiti a nenhuma criatura o legado da nossa miséria',
    'Ao verme que primeiro roeu as frias carnes do meu cadáver dedico como saudosa lembrança estas memórias póstumas'
]

In [9]:
for prompt in prompts_machado:
    context = torch.tensor(encode(prompt), dtype=torch.long, device=device)
    generated_chars = decode(
        m.generate(context.unsqueeze(0), max_new_tokens=700, temperature=0.8, top_k=40)[0].tolist()
    )

    print(f'\n{'=' * 80}\nPROMPT: {prompt}\n{'=' * 80}')
    print(generated_chars)


PROMPT: Ao vencedor, as batatas
Ao vencedor, as batatas fazem suspender de uma redenção espiritual com
unhas de outra predição de engeria, e outras trazem ao adágio das obras de uma
proibição. Se estas reunirem as aparências da receita de um congresso mais difícil
do congresso as estátuas sociais, não se alimenta nem preva, como condena o
inverno dos costumes e dos pensadores, e com raro curiosidade a vida de
respeito e respeito em ambas as instituições amadas.
CAPÍTULO III
Um mestre de consolação mais resulta que o problema de alguns meses pode acudir
aos solambentos das coisas; mas não eram escritos no país e no Hotel de
Caracas, e para ele a coisa precisa de ledicos apropriados. Uma crise se há, pode
crer que as coisas estão prontas e as sens

PROMPT: Deus te livre, leitor, de uma ideia fixa
Deus te livre, leitor, de uma ideia fixa, e que és tu que não és
Vulcano, que não ri, missão de graça, de um desejo. O mistério, ignoro
a força íntegra e pura incorrerás nas seguintes velas, da

In [10]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=500)[0].tolist()))

 mestre do conteato que vinha tocar no horizonte, o radil de
Lacerda extremamente lânguida, como a verdadeira razão da forma
principal, e antes de tudo isso me era preciso acabar com o desejo
de ver o resguardo das graças do papel, que se esqueceram da
inteligência moral. Entendi o que era a câmara boa principal;
depois vi que, em consciência, a desforra da reclamação, da parte
dele na própria câmara, foram as próprias obras do país, e na
proposta da fatal notícia. Tal foi a imprensa com mais côm
